# TPG speaker video — corrected layout (v2)

Keeps the sample-video logo/title and right-side elephant in their original locations. The speaker is scaled smaller and lowered so the headings remain visible. The covered elephant cannot be recovered from the reference alone; supply a clean background PNG for a perfect reconstruction.


In [ ]:
!git clone -q --depth 1 https://github.com/PeterL1n/RobustVideoMatting.git /content/RobustVideoMatting
!pip -q install opencv-python-headless pillow tqdm
!apt-get -qq update >/dev/null && apt-get -qq install -y ffmpeg >/dev/null


## Upload original video, reference video, and optionally a clean background image

The longest video is treated as the original; the shorter video is the reference. You can also upload a clean 1280×720 background image for the best quality.

In [ ]:
from google.colab import files
from pathlib import Path
import subprocess, json, os
uploaded=files.upload()
items=[Path(x) for x in uploaded]
vids=sorted([x for x in items if x.suffix.lower() in ('.mp4','.mov','.mkv')],key=lambda x:float(json.loads(subprocess.check_output(['ffprobe','-v','error','-show_entries','format=duration','-of','json',str(x)]))['format']['duration']),reverse=True)
imgs=[x for x in items if x.suffix.lower() in ('.jpg','.jpeg','.png')]
assert len(vids)>=2,'Upload both original (100s) and reference (15s) videos.'
INPUT=str(vids[0]); REFERENCE=str(vids[1]); BACKGROUND=str(imgs[0]) if imgs else None
print('Original:',INPUT,'Reference:',REFERENCE,'Clean background:',BACKGROUND or 'none — will reconstruct')

## Load pretrained RVM weights
Model weights are downloaded from the official RVM GitHub release. Internet access is needed once. GPU is recommended; CPU processing of 100 seconds can be very slow.

In [ ]:
import sys, torch, urllib.request
sys.path.insert(0,'/content/RobustVideoMatting')
from model import MattingNetwork
WEIGHTS='/content/rvm_mobilenetv3.pth'
if not os.path.exists(WEIGHTS):
    urllib.request.urlretrieve('https://github.com/PeterL1n/RobustVideoMatting/releases/download/v1.0.0/rvm_mobilenetv3.pth', WEIGHTS)
device='cuda' if torch.cuda.is_available() else 'cpu'
model=MattingNetwork('mobilenetv3').eval().to(device)
model.load_state_dict(torch.load(WEIGHTS,map_location=device,weights_only=True))
print('Model ready on',device)


## Layout settings

Adjust the speaker position after previewing. The speaker is centered between the logo at upper left and the title at upper right. If the speaker overlaps the logo, reduce SPEAKER_SCALE.

In [ ]:
import cv2, numpy as np
from PIL import Image
from IPython.display import display
W,H=1280,720
def frame_at(path,seconds=1):
    cap=cv2.VideoCapture(path);cap.set(cv2.CAP_PROP_POS_MSEC,seconds*1000)
    ok,fr=cap.read();cap.release()
    if not ok:raise RuntimeError(f'Could not read {path} at {seconds}s')
    return cv2.cvtColor(fr,cv2.COLOR_BGR2RGB)
ref= cv2.resize(frame_at(REFERENCE,2),(W,H))
display(Image.fromarray(ref).resize((640,360)))

In [ ]:
# Speaker is reduced and lowered to leave room for BOTH headings.
SPEAKER_SCALE=0.78
SPEAKER_SHIFT_X=0
SPEAKER_SHIFT_Y=105
DOWNSAMPLE_RATIO=0.5
INPAINT_DILATION=15
# Preserve exact reference artwork in these areas (1280x720 coordinates).
HEADER_PROTECT_Y=155
ELEPHANT_PROTECT_X=890
# Change speaker scale/shift only after checking preview.


## Build protected reference background and preview

The upper branding strip and right elephant region are preserved from the reference. The reference speaker is removed only from the central region; inpainting can produce artifacts where the original speaker covered artwork.


In [ ]:
import torch.nn.functional as F
import torch

def matte(rgb,rec=None):
    src=torch.from_numpy(np.ascontiguousarray(rgb)).permute(2,0,1).unsqueeze(0).float().to(device)/255
    with torch.no_grad():
        fgr,pha,*newrec=model(src,*(rec if rec is not None else [None]*4),downsample_ratio=DOWNSAMPLE_RATIO)
    f=(fgr[0].permute(1,2,0).cpu().numpy()*255).clip(0,255).astype(np.uint8)
    a=pha[0,0].cpu().numpy().clip(0,1)
    return f,a,newrec

if BACKGROUND:
    bg=np.array(Image.open(BACKGROUND).convert('RGB').resize((W,H),Image.Resampling.LANCZOS))
else:
    # Inpaint the REFERENCE speaker only in the middle. Do not inpaint
    # the top branding or the right-hand elephant artwork.
    _,reference_alpha,_=matte(ref)
    mask=(reference_alpha>0.28).astype(np.uint8)*255
    mask=cv2.dilate(mask,np.ones((INPAINT_DILATION,INPAINT_DILATION),np.uint8))
    mask[:HEADER_PROTECT_Y,:]=0
    mask[:,ELEPHANT_PROTECT_X:]=0
    # The protected areas retain the exact pixels from the sample video.
    bg=cv2.cvtColor(cv2.inpaint(cv2.cvtColor(ref,cv2.COLOR_RGB2BGR),mask,7,cv2.INPAINT_TELEA),cv2.COLOR_BGR2RGB)


def compose(f,a):
    # Resize entire original frame's foreground and alpha together.
    th=int(round(H*SPEAKER_SCALE));tw=int(round(W*SPEAKER_SCALE))
    f=cv2.resize(f,(tw,th),interpolation=cv2.INTER_LINEAR)
    a=cv2.resize(a,(tw,th),interpolation=cv2.INTER_LINEAR)
    x=(W-tw)//2+SPEAKER_SHIFT_X;y=(H-th)//2+SPEAKER_SHIFT_Y
    out=bg.astype(np.float32).copy()
    x0=max(0,x);y0=max(0,y);x1=min(W,x+tw);y1=min(H,y+th)
    if x1>x0 and y1>y0:
        fg=f[y0-y:y1-y,x0-x:x1-x].astype(np.float32)
        alpha=a[y0-y:y1-y,x0-x:x1-x,None].astype(np.float32)
        out[y0:y1,x0:x1]=fg*alpha+out[y0:y1,x0:x1]*(1-alpha)
    return np.clip(out,0,255).astype(np.uint8)

f,a,_=matte(cv2.resize(frame_at(INPUT,2),(W,H)))
preview=compose(f,a)
Image.fromarray(bg).save('/content/tpg_reconstructed_background.png')
Image.fromarray(preview).save('/content/tpg_preview.png')
display(Image.fromarray(preview).resize((960,540)))

## Render all 100 seconds with recurrent RVM
RVM maintains recurrent state across frames. This is slower than a fixed mask but much more appropriate for a moving speaker. A temporary silent MP4 is produced first, then FFmpeg copies the original audio into the final MP4. **Do not close the Colab session during rendering.**

In [ ]:
from tqdm.auto import tqdm
cap=cv2.VideoCapture(INPUT)
fps=cap.get(cv2.CAP_PROP_FPS) or 30
count=int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
assert cap.isOpened() and count>0
silent='/content/tpg_silent.mp4'
writer=cv2.VideoWriter(silent,cv2.VideoWriter_fourcc(*'mp4v'),fps,(W,H))
assert writer.isOpened(),'Unable to create output MP4'
rec=[None]*4
try:
    for i in tqdm(range(count),desc='Rendering complete TPG video'):
        ok,frame=cap.read()
        if not ok:break
        rgb=cv2.cvtColor(cv2.resize(frame,(W,H)),cv2.COLOR_BGR2RGB)
        f,a,rec=matte(rgb,rec)
        result=compose(f,a)
        writer.write(cv2.cvtColor(result,cv2.COLOR_RGB2BGR))
finally:
    cap.release();writer.release()
FINAL='/content/TPG_Teacher_Candidate_Final_100s.mp4'
cmd=['ffmpeg','-y','-loglevel','error','-i',silent,'-i',INPUT,'-map','0:v:0','-map','1:a:0?','-c:v','libx264','-preset','veryfast','-crf','18','-pix_fmt','yuv420p','-c:a','aac','-b:a','192k','-shortest',FINAL]
subprocess.run(cmd,check=True)
print('Final video:',FINAL)

## Verify and download
Inspect the video at 0, 15, 30, 60, 90 and 99 seconds before calling it final. RVM can still struggle with similar foreground/background colors, rapid movement and fine hair. If there are defects, refine the matte or try a different model; don't simply blur the boundary.

In [ ]:
from IPython.display import Video,display
print(subprocess.check_output(['ffprobe','-v','error','-show_entries','format=duration','-of','default=nw=1',FINAL]).decode())
display(Video(FINAL,embed=False,width=640))
files.download(FINAL)